# Fine-tune ViT5-base + LoRA for Vietnamese News Summarization

This notebook is intentionally simple: all important training values are shown directly here instead of being loaded from external YAML config. The values match the historical dataset v2 report.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate peft rouge-score sentencepiece torch pandas numpy bert-score

## 2. Imports and training parameters

In [ ]:
from pathlib import Path
import numpy as np
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

DATASET_DIR = Path('data/datasets/v2')
MODEL_NAME = 'VietAI/vit5-base'
OUTPUT_DIR = 'models/vit5-news-v2'

MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
GENERATION_NUM_BEAMS = 4
SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q', 'v']

## 3. Load dataset v2

Expected historical split sizes: 1636 train / 218 validation / 216 test.

In [ ]:
dataset_files = {
    'train': str(DATASET_DIR / 'train.jsonl'),
    'validation': str(DATASET_DIR / 'val.jsonl'),
    'test': str(DATASET_DIR / 'test.jsonl'),
}
raw_ds = load_dataset('json', data_files=dataset_files)
raw_ds

## 4. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    model_inputs = tokenizer(
        batch['content_text'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['summary'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_ds = raw_ds.map(preprocess, batched=True, remove_columns=raw_ds['train'].column_names)
tokenized_ds

## 5. Build ViT5 + LoRA model

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 6. Metrics

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.asarray(preds)
    labels = np.asarray(labels)
    pad_id = tokenizer.pad_token_id
    preds = np.where(preds < 0, pad_id, preds)
    labels = np.where(labels < 0, pad_id, labels)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    totals = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    for pred, ref in zip(decoded_preds, decoded_labels):
        scores = scorer.score(ref, pred)
        for key in totals:
            totals[key] += scores[key].fmeasure
    n = max(len(decoded_preds), 1)
    return {key: value / n for key, value in totals.items()}


def compute_rouge_list(predictions, references):
    totals = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        for key in totals:
            totals[key] += scores[key].fmeasure
    n = max(len(predictions), 1)
    return {key: round(value / n, 4) for key, value in totals.items()}

## 7. Train

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    predict_with_generate=True,
    generation_num_beams=GENERATION_NUM_BEAMS,
    generation_max_length=MAX_TARGET_LENGTH,
    logging_steps=50,
    save_total_limit=2,
    seed=SEED,
    report_to=[],
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## 8. Evaluate test split and save

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=tokenized_ds['test'], metric_key_prefix='test')
test_metrics

# Historical report values for the original v2 run:
# ROUGE-1 F1 = 0.6055, ROUGE-2 F1 = 0.3106, ROUGE-L F1 = 0.3804, loss = 1.4121

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 9. Optional eval: LexRank, TextRank, ViT5, BERTScore

The simplified runtime does not require MLflow. This section keeps the historical eval intent: compare LexRank, TextRank, and fine-tuned ViT5 against the Gemini teacher references, then optionally compute BERTScore.

In [ ]:
# Optional extractive baselines + BERTScore install:
# !pip install -q sumy bert-score
# from sumy.parsers.plaintext import PlaintextParser
# from sumy.nlp.tokenizers import Tokenizer
# from sumy.summarizers.lex_rank import LexRankSummarizer
# from sumy.summarizers.text_rank import TextRankSummarizer

# def extractive_summary(text, summarizer_cls, n_sentences=3):
#     parser = PlaintextParser.from_string(text, Tokenizer('english'))
#     summarizer = summarizer_cls()
#     return ' '.join(str(sentence) for sentence in summarizer(parser.document, n_sentences))

# lexrank_predictions = [extractive_summary(row['content_text'], LexRankSummarizer) for row in raw_ds['test']]
# textrank_predictions = [extractive_summary(row['content_text'], TextRankSummarizer) for row in raw_ds['test']]
# references = [row['summary'] for row in raw_ds['test']]
# print('lexrank', compute_rouge_list(lexrank_predictions, references))
# print('textrank', compute_rouge_list(textrank_predictions, references))

# from bert_score import score
# predictions = trainer.predict(tokenized_ds['test']).predictions
# predictions = np.where(np.asarray(predictions) < 0, tokenizer.pad_token_id, predictions)
# decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
# decoded_references = [row['summary'] for row in raw_ds['test']]
# _, _, bert_f1 = score(decoded_predictions, decoded_references, lang='vi', model_type='xlm-roberta-base')
# float(bert_f1.mean())

## 10. Optional push to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub('your-username/vit5-news-v2')
# tokenizer.push_to_hub('your-username/vit5-news-v2')